### 1st code 

In [2]:
# ===========================
# 1. IMPORTS
# ===========================
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input


# ===========================
# 2. LOAD DATA
# ===========================
X_images = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_images.npy")
X_rna = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_rna.npy")
X_clinical = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_clinical.npy")
y = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\y.npy")

print("Loaded:", X_images.shape, X_rna.shape, X_clinical.shape)


# ===========================
# 3. IMAGE FEATURES (CACHE)
# ===========================
cache_path = "X_img_features.npy"

if os.path.exists(cache_path):
    print("Loading cached image features...")
    X_img_features = np.load(cache_path)

else:
    print("Extracting image features...")

    n_patients, n_slices = X_images.shape[0], X_images.shape[1]

    X_img = X_images.reshape(-1, 224, 224, 1)
    X_img = np.repeat(X_img, 3, axis=-1)
    X_img = preprocess_input(X_img)

    base_model = MobileNetV3Small(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )
    base_model.trainable = False

    features = base_model.predict(X_img, verbose=1)

    features = np.mean(features, axis=(1,2))
    features = features.reshape(n_patients, n_slices, -1)

    X_img_features = np.mean(features, axis=1)

    np.save(cache_path, X_img_features)
    print("Saved image features!")


# ===========================
# 4. MODEL
# ===========================
class BetterNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# ===========================
# 5. CROSS VALIDATION
# ===========================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
f1_scores = []
auc_scores = []
inference_times = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_img_features, y)):
    print(f"\n===== FOLD {fold+1} =====")

    # split
    X_img_tr, X_img_te = X_img_features[train_idx], X_img_features[test_idx]
    X_rna_tr, X_rna_te = X_rna[train_idx], X_rna[test_idx]
    X_clin_tr, X_clin_te = X_clinical[train_idx], X_clinical[test_idx]

    y_train, y_test = y[train_idx], y[test_idx]

    # ===========================
    # PCA (NO LEAKAGE)
    # ===========================
    pca_img = PCA(n_components=20)
    X_img_tr = pca_img.fit_transform(X_img_tr)
    X_img_te = pca_img.transform(X_img_te)

    pca_rna = PCA(n_components=20)
    X_rna_tr = pca_rna.fit_transform(X_rna_tr)
    X_rna_te = pca_rna.transform(X_rna_te)

    # ===========================
    # FUSION
    # ===========================
    X_train = np.concatenate([X_img_tr, X_rna_tr, X_clin_tr], axis=1)
    X_test = np.concatenate([X_img_te, X_rna_te, X_clin_te], axis=1)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # torch
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    model = BetterNN(X_train.shape[1])

    # imbalance handling (IMPORTANT)
    pos_weight = torch.tensor([(len(y_train) - sum(y_train)) / sum(y_train)])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # ===========================
    # TRAIN
    # ===========================
    for epoch in range(80):
        model.train()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # ===========================
    # EVALUATION + INFERENCE TIME
    # ===========================
    model.eval()

    runs = 50
    total_time = 0

    with torch.no_grad():
        for _ in range(runs):
            start = time.perf_counter()
            logits = model(X_test_t)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            end = time.perf_counter()
            total_time += (end - start)

        preds = torch.sigmoid(logits).numpy()

    inference_time = total_time / (runs * len(X_test_t))
    inference_times.append(inference_time)

    print(f"Inference Time per sample: {inference_time:.6f} sec")

    # ===========================
    # THRESHOLD TUNING (RESTORED)
    # ===========================
    best_f1 = 0
    best_thresh = 0.5

    for t in np.arange(0.1, 0.9, 0.05):
        temp_preds = (preds > t).astype(int)
        f1 = f1_score(y_test, temp_preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    final_preds = (preds > best_thresh).astype(int)

    acc = accuracy_score(y_test, final_preds)
    f1 = f1_score(y_test, final_preds)
    auc = roc_auc_score(y_test, preds)

    print(f"Best Threshold: {best_thresh:.2f}")
    print(f"Accuracy: {acc:.3f}, F1: {f1:.3f}, AUC: {auc:.3f}")

    acc_scores.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)


# ===========================
# FINAL RESULTS
# ===========================
print("\n===== FINAL RESULTS =====")
print("Mean Accuracy:", np.mean(acc_scores))
print("Mean F1 Score:", np.mean(f1_scores))
print("Mean AUC:", np.mean(auc_scores))


# ===========================
# INFERENCE SUMMARY
# ===========================
print("\n===== INFERENCE TIME SUMMARY =====")
print("Mean:", np.mean(inference_times))
print("Min:", np.min(inference_times))
print("Max:", np.max(inference_times))

Loaded: (42, 50, 224, 224, 1) (42, 20) (42, 76)
Extracting image features...
66/66 ━━━━━━━━━━━━━━━━━━━━ 19s 247ms/step
Saved image features!

===== FOLD 1 =====
Inference Time per sample: 0.000392 sec
Best Threshold: 0.15
Accuracy: 0.889, F1: 0.857, AUC: 1.000

===== FOLD 2 =====
Inference Time per sample: 0.000024 sec
Best Threshold: 0.70
Accuracy: 0.778, F1: 0.667, AUC: 0.611

===== FOLD 3 =====
Inference Time per sample: 0.000025 sec
Best Threshold: 0.10
Accuracy: 0.500, F1: 0.500, AUC: 0.333

===== FOLD 4 =====
Inference Time per sample: 0.000025 sec
Best Threshold: 0.10
Accuracy: 0.500, F1: 0.500, AUC: 0.583

===== FOLD 5 =====
Inference Time per sample: 0.000031 sec
Best Threshold: 0.60
Accuracy: 0.875, F1: 0.800, AUC: 0.833

===== FINAL RESULTS =====
Mean Accuracy: 0.7083333333333333
Mean F1 Score: 0.6647619047619047
Mean AUC: 0.6722222222222223

===== INFERENCE TIME SUMMARY =====
Mean: 9.952976666643722e-05
Min: 2.4407555555272767e-05
Max: 0.00039209377777814954


### 2nd code 

In [8]:
# ===========================
# 1. IMPORTS
# ===========================
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input


# ===========================
# 2. LOAD DATA
# ===========================
X_images = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_images.npy")
X_rna = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_rna.npy")
X_clinical = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_clinical.npy")
y = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\y.npy")

print("Loaded:", X_images.shape, X_rna.shape, X_clinical.shape)


# ===========================
# 3. IMAGE FEATURES (CACHE)
# ===========================
cache_path = "X_img_features.npy"

if os.path.exists(cache_path):
    print("Loading cached image features...")
    X_img_features = np.load(cache_path)

else:
    print("Extracting image features...")

    n_patients, n_slices = X_images.shape[0], X_images.shape[1]

    X_img = X_images.reshape(-1, 224, 224, 1)
    X_img = np.repeat(X_img, 3, axis=-1)
    X_img = preprocess_input(X_img)

    base_model = MobileNetV3Small(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )
    base_model.trainable = False

    features = base_model.predict(X_img, verbose=1)

    features = np.mean(features, axis=(1,2))
    features = features.reshape(n_patients, n_slices, -1)

    X_img_features = np.mean(features, axis=1)

    np.save(cache_path, X_img_features)
    print("Saved image features!")


# ===========================
# 4. MODEL
# ===========================
class BetterNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# ===========================
# 5. CROSS VALIDATION
# ===========================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
f1_scores = []
auc_scores = []
inference_times = []
best_thresholds = []   # 🔥 NEW

for fold, (train_idx, test_idx) in enumerate(kf.split(X_img_features, y)):
    print(f"\n===== FOLD {fold+1} =====")

    # split
    X_img_tr, X_img_te = X_img_features[train_idx], X_img_features[test_idx]
    X_rna_tr, X_rna_te = X_rna[train_idx], X_rna[test_idx]
    X_clin_tr, X_clin_te = X_clinical[train_idx], X_clinical[test_idx]

    y_train, y_test = y[train_idx], y[test_idx]

    # PCA
    pca_img = PCA(n_components=20)
    X_img_tr = pca_img.fit_transform(X_img_tr)
    X_img_te = pca_img.transform(X_img_te)

    pca_rna = PCA(n_components=20)
    X_rna_tr = pca_rna.fit_transform(X_rna_tr)
    X_rna_te = pca_rna.transform(X_rna_te)

    # fusion
    X_train = np.concatenate([X_img_tr, X_rna_tr, X_clin_tr], axis=1)
    X_test = np.concatenate([X_img_te, X_rna_te, X_clin_te], axis=1)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # torch
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    model = BetterNN(X_train.shape[1])

    # imbalance handling
    pos_weight = torch.tensor([(len(y_train) - sum(y_train)) / sum(y_train)])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # TRAIN
    for epoch in range(80):
        model.train()
        loss = criterion(model(X_train_t), y_train_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # INFERENCE TIMING
    model.eval()
    runs = 50
    total_time = 0

    with torch.no_grad():
        for _ in range(runs):
            start = time.perf_counter()
            logits = model(X_test_t)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            end = time.perf_counter()
            total_time += (end - start)

        preds = torch.sigmoid(logits).numpy()

    inference_time = total_time / (runs * len(X_test_t))
    inference_times.append(inference_time)

    print(f"Inference Time per sample: {inference_time:.6f} sec")

    # THRESHOLD TUNING
    best_f1 = 0
    best_thresh = 0.5

    for t in np.arange(0.1, 0.9, 0.05):
        temp_preds = (preds > t).astype(int)
        f1 = f1_score(y_test, temp_preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    best_thresholds.append(best_thresh)   # 🔥 STORE

    final_preds = (preds > best_thresh).astype(int)

    acc = accuracy_score(y_test, final_preds)
    f1 = f1_score(y_test, final_preds)
    auc = roc_auc_score(y_test, preds)

    print(f"Best Threshold: {best_thresh:.2f}")
    print(f"Accuracy: {acc:.3f}, F1: {f1:.3f}, AUC: {auc:.3f}")

    acc_scores.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)


# ===========================
# 6. GLOBAL THRESHOLD
# ===========================
global_threshold = np.mean(best_thresholds)
print("\nGlobal Threshold:", round(global_threshold, 3))


# ===========================
# 7. FINAL RESULTS
# ===========================
print("\n===== FINAL RESULTS =====")
print("Mean Accuracy:", np.mean(acc_scores))
print("Mean F1 Score:", np.mean(f1_scores))
print("Mean AUC:", np.mean(auc_scores))


# ===========================
# 8. INFERENCE SUMMARY
# ===========================
print("\n===== INFERENCE TIME SUMMARY =====")
print("Mean:", np.mean(inference_times))
print("Min:", np.min(inference_times))
print("Max:", np.max(inference_times))

Loaded: (42, 50, 224, 224, 1) (42, 20) (42, 76)
Loading cached image features...

===== FOLD 1 =====
Inference Time per sample: 0.000049 sec
Best Threshold: 0.45
Accuracy: 1.000, F1: 1.000, AUC: 1.000

===== FOLD 2 =====
Inference Time per sample: 0.000014 sec
Best Threshold: 0.65
Accuracy: 0.778, F1: 0.500, AUC: 0.444

===== FOLD 3 =====
Inference Time per sample: 0.000017 sec
Best Threshold: 0.10
Accuracy: 0.375, F1: 0.444, AUC: 0.267

===== FOLD 4 =====
Inference Time per sample: 0.000017 sec
Best Threshold: 0.20
Accuracy: 0.625, F1: 0.571, AUC: 0.708

===== FOLD 5 =====
Inference Time per sample: 0.000017 sec
Best Threshold: 0.50
Accuracy: 0.750, F1: 0.667, AUC: 0.750

Global Threshold: 0.38

===== FINAL RESULTS =====
Mean Accuracy: 0.7055555555555555
Mean F1 Score: 0.6365079365079365
Mean AUC: 0.6338888888888888

===== INFERENCE TIME SUMMARY =====
Mean: 2.311073888841975e-05
Min: 1.408288888822224e-05
Max: 4.932155555454503e-05
